# Mini-Project APM_5AI29 : Text 2 SQL

__Team: P2D2 (Prompt 2 Data-Demand)__
_Members:_
* _Jeanne Malécot_
* _Arthur Nuvoloni_
* _Adam Sebti_
* _Benjamin Ternot_

Using `meta-llama/Llama-3.2-1B`model

> Dataset : https://huggingface.co/datasets/xlangai/spider

## **Text-to SQL**

In [1]:
import copy
import os

import evaluate
import numpy as np
import random
from tensorflow.python.client import device_lib
import torch

from data.databases import DBInfos, format_schema

2025-01-06 17:34:12.038902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736181252.059983 1143141 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736181252.067122 1143141 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-06 17:34:12.090441: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
os.environ["TORCH_USE_CUDA_DSA"]="1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # Add this line to enable more detailed error messages
#print(device_lib.list_local_devices())

# Dataset 

In [3]:
# DL dataset
from datasets import load_dataset
ds = load_dataset("xlangai/spider")

In [4]:
train_set = ds["train"]
validation_set = ds["validation"]

# split the validation set into new validation and test sets
val_set = validation_set.select(range(0,1000))
test_set = validation_set.select(range(1000, len(validation_set)))

print(train_set)
print(val_set)
print(test_set)

Dataset({
    features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
    num_rows: 7000
})
Dataset({
    features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
    num_rows: 1000
})
Dataset({
    features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
    num_rows: 34
})


In [5]:
#print one row
row_id = 5

for key in train_set[row_id]:
    print(f"{key}:\t{train_set[row_id][key]}")

db_id:	department_management
query:	SELECT name FROM head WHERE born_state != 'California'
question:	What are the names of the heads who are born outside the California state?
query_toks:	['SELECT', 'name', 'FROM', 'head', 'WHERE', 'born_state', '!', '=', "'California", "'"]
query_toks_no_value:	['select', 'name', 'from', 'head', 'where', 'born_state', '!', '=', 'value']
question_toks:	['What', 'are', 'the', 'names', 'of', 'the', 'heads', 'who', 'are', 'born', 'outside', 'the', 'California', 'state', '?']


In [6]:
db_infos = DBInfos("data/tables.json")

# Model

### **Llama-3.2-1B:**  
Hugging Face, llama-base: https://huggingface.co/meta-llama/Llama-3.2-1B

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig 

# quantization_config = BitsAndBytesConfig(
#     load_in_8bit=False, load_in_4bit=True
# )

name = 'meta-llama/Llama-3.2-1B'

with open('huggingface_token.txt', 'r') as f:
    tokens=[line for line in f]
hf_token = tokens[0]

tokenizer = AutoTokenizer.from_pretrained(name, token = hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    name, 
    # quantization_config=quantization_config, 
    device_map={"": 0},
    torch_dtype=torch.bfloat16, 
    token = hf_token
)
model.config.pad_token_id = tokenizer.pad_token_id

device = model.device

model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm):

>We will at first use prompt-engineering methods to try to perform the text-2-sql conversion, without any fine-tuning.

# Test without Fine-tuning

## ZERO SHOT 

In [8]:
example = train_set[0]
question = example['question']
gold_sql = example['query']
print(question)
print(">>>", gold_sql)

How many heads of the departments are older than 56 ?
>>> SELECT count(*) FROM head WHERE age  >  56


In [9]:
def generate(prompt, model, tokenizer):
    device = model.device

    # Tokenize input with attention mask
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        max_length=3000, 
        truncation=True, 
        padding=True
    ).to(device)

    prompt_length = inputs['input_ids'].shape[1]

    # Generate output
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        temperature=0.1,
        top_k=50,
        top_p=0.85,
    )

    # Decode generated text
    sql_query = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return sql_query

### Basic prompt

>We want to convert this example question to an SQL Query.  
Let's use directly `Llama-3.2-1B`, without specific informations about the databases.

In [10]:
zeroshot_prompt = """
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.

You will only answer with the SQL Query.

<<<
Question: {question}
SQL Query:
>>>
""".strip()

>Test the code:

In [11]:
prompt = zeroshot_prompt.format(question=question)
answer = generate(prompt, model, tokenizer)

print("##### Example 0 #####")
print(prompt)
print("-" * 20)
print(answer)
print("-" * 20)
print(gold_sql)

##### Example 0 #####
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.

You will only answer with the SQL Query.

<<<
Question: How many heads of the departments are older than 56 ?
SQL Query:
>>>
--------------------
 SELECT COUNT(*) FROM department WHERE age > '56'
<<<

--------------------
SELECT count(*) FROM head WHERE age  >  56


### Detailed prompt

> This methods is promising, but the models can make errors on table or columns names or datatypes. We will try with a more detailed prompt, based on the informations that we retrieved from the databases (Schema...) in order to make sure that the model understands the table and columns.

In [12]:
db_id = example['db_id']
infos = db_infos.get(db_id)

schemas, p_keys, f_keys = format_schema(infos)

In [13]:
zeroshot_detail_prompt = """
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

You are working with the database {db_id}, with the following informations:

Schemas:
{schemas}

Primary Keys:
{p_keys}

Foreign Keys:
{f_keys}

You will only answer with the SQL Query.

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [14]:
prompt = zeroshot_detail_prompt.format(
    question=question, 
    db_id=db_id,
    schemas=schemas,
    p_keys=p_keys,
    f_keys=f_keys
)
answer = generate(prompt, model, tokenizer)

print("##### Example 0 #####")
print(prompt)
print("-" * 20)
print(answer)
print("-" * 20)
print(gold_sql)

##### Example 0 #####
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

You are working with the database department_management, with the following informations:

Schemas:
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
head : head_ID (number) , name (text) , born_state (text) , age (number)
management : department_ID (number) , head_ID (number) , temporary_acting (text)

Primary Keys:
department : Department_ID
head : head_ID
management : department_ID

Foreign Keys:
management : head_ID equals head : head_ID
management : department_ID equals department : Department_ID

You will only answer with the SQL Query.

<<<
Question: How many heads of the departmen

> This zeroshot method still produces hallucinations on column names with `Llama`.

## FEW SHOT

In [15]:
def format_demo(demo):
    return f"Question: {demo['question']}\nSQL Query: {demo['query']}."
    
def format_demos(demos):
    formatted_str = []
    for demo in demos:
        formatted_str.append(format_demo(demo))
    return '\n\n'.join(formatted_str)
        
def get_random_demo(k, train=train_set):
    examples = []
    for i in range(k):
        examples.append(random.choice(train))
    return examples

### Basic Prompt

In [16]:
fewshot_prompt = """
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.

You will only answer with the SQL Query.

####
Here are some examples:

{examples}
####

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [17]:
demos = format_demos(get_random_demo(5))

prompt = fewshot_prompt.format(examples=demos, question=question)
answer = generate(prompt, model, tokenizer)

print("##### Example 0 #####")
print(prompt)
print("-" * 20)
print(answer)
print("-" * 20)
print(gold_sql)

##### Example 0 #####
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.

You will only answer with the SQL Query.

####
Here are some examples:

Question: What is the name of the course that has the most student enrollment?
SQL Query: SELECT T1.course_name FROM Courses AS T1 JOIN Student_Course_Enrolment AS T2 ON T1.course_id  =  T2.course_id GROUP BY T1.course_name ORDER BY COUNT(*) DESC LIMIT 1.

Question: Return the weight of the shortest person.
SQL Query: SELECT Weight FROM people ORDER BY Height ASC LIMIT 1.

Question: Find Alice's friends of friends.
SQL Query: SELECT DISTINCT T4.name FROM PersonFriend AS T1 JOIN Person AS T2 ON T1.name  =  T2.name JOIN PersonFriend AS T3 ON T1.friend  =  T3.name JOIN PersonFriend AS T4 ON T3.friend  =  T4.name WHERE T2.name  =  'Alice' AND T4.name != 'Alice'.

Question: Count the number of courses.
SQL Query: SELECT count(*) FROM COURSE.

Question: Show the lieutenant governor and comptrolle

### Detailed Prompt

In [18]:
fewshot_detail_prompt = """
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

You are working with the database {db_id}, with the following informations:

Schemas:
{schemas}

Primary Keys:
{p_keys}

Foreign Keys:
{f_keys}

You will only answer with the SQL Query.

####
Here are some examples:

{examples}
####

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [19]:
demos = format_demos(get_random_demo(5))

prompt = fewshot_detail_prompt.format(
    db_id=db_id,
    schemas=schemas,
    p_keys=p_keys,
    f_keys=f_keys,
    examples=demos,
    question=question, 
)
answer = generate(prompt, model, tokenizer)

print("##### Example 0 #####")
print(prompt)
print("-" * 20)
print(answer)
print("-" * 20)
print(gold_sql)

##### Example 0 #####
You are an expert in databases. Your task is to convert question between <<<>>> into a valid SQL Query.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

You are working with the database department_management, with the following informations:

Schemas:
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
head : head_ID (number) , name (text) , born_state (text) , age (number)
management : department_ID (number) , head_ID (number) , temporary_acting (text)

Primary Keys:
department : Department_ID
head : head_ID
management : department_ID

Foreign Keys:
management : head_ID equals head : head_ID
management : department_ID equals department : Department_ID

You will only answer with the SQL Query.

####
Here are some examples:

Question: How m

# FINE TUNING

> Prepare the dataset for fine-tuning:

In [20]:
from datasets import Dataset

train_data_dict = {
    "input": train_set['question'],
    "target": train_set['query']
}

train_dataset = Dataset.from_dict(train_data_dict)

val_data_dict = {
    "input": val_set['question'],
    "target": val_set['query']
}

val_dataset = Dataset.from_dict(val_data_dict)

In [22]:
from peft import LoraConfig, PeftModel, get_peft_model

model = AutoModelForCausalLM.from_pretrained(name, load_in_8bit=True, device_map="auto", token=hf_token)

# Define LoRA Config
lora_config = LoraConfig(
    r=8, # Rank of the LoRA update matrices
    lora_alpha=32, # Scaling factor for the LoRA updates
    lora_dropout=0.05, # Dropout probability for the LoRA layers
    bias="none",  # Whether to apply a bias to the LoRA layers
    task_type="SEQ_2_SEQ_LM"  # Type of task for which the model is being fine-tuned
)

# Apply LoRA to the model using peft
ft_model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [23]:
# Preprocessing function
def preprocess_function(examples):
    inputs = [f"Question: {q}\nSQL: " for q in examples["input"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length') # Add padding='max_length'

    with tokenizer.as_target_tokenizer():
      labels = tokenizer(examples["target"], max_length=512, truncation=True, padding='max_length') # Add padding='max_length'

    # Create attention mask: 1 for real tokens, 0 for padding
    model_inputs["attention_mask"] = [[1] * len(input_ids) + [0] * (512 - len(input_ids)) for input_ids in model_inputs['input_ids']]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_datasets = train_dataset.map(preprocess_function, batched=True)
tokenized_val_datasets = val_dataset.map(preprocess_function, batched=True)

# Define the metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Calculate the metrics
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"accuracy": result["accuracy"]}

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

/home/infres/bternot-21/Text2SQL/venv-Text2SQL/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

>Training arguments:

In [24]:
from transformers import TrainingArguments, Trainer

# Define training arguments
training_args = TrainingArguments(
    output_dir="llama3-sql-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_accumulation_steps=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=10,
    push_to_hub=False,
    fp16=True,
    dataloader_num_workers=4,  # Use multiple workers for faster data loading
    report_to="none",
    ddp_find_unused_parameters=False,  # Helps with Distributed Data Parallel (DDP)
)

# Define trainer
trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train_datasets,
    eval_dataset=tokenized_val_datasets,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/home/infres/bternot-21/Text2SQL/venv-Text2SQL/lib/python3.12/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_1143141/809371524.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Fine-tune the model
#os.environ['TOKENIZERS_PARALLELISM']='true'
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch,Training Loss,Validation Loss


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [48]:
ft_model.save_pretrained("./fine_tuned_llama")
tokenizer.save_pretrained("./fine_tuned_llama")

('./fine_tuned_bart/tokenizer_config.json',
 './fine_tuned_bart/special_tokens_map.json',
 './fine_tuned_bart/vocab.json',
 './fine_tuned_bart/merges.txt',
 './fine_tuned_bart/added_tokens.json')

In [56]:
model_path = "./fine_tuned_llama"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Inference
input_text = "How many heads of the departments are older than 56 ?"
inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)
outputs = model.generate(inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True)

# Decode and print the SQL query
sql_query = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(sql_query)

SELECT count(*) FROM department WHERE age  >  56


## ZERO SHOT 

In [57]:
example = train_set[0]
question = example['question']
gold_sql = example['query']
print(question)
print(">>>", gold_sql)

How many heads of the departments are older than 56 ?
>>> SELECT count(*) FROM head WHERE age  >  56


In [58]:
def generate(prompt, model=model, tokenizer=tokenizer):

    device = model.device
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=3000, truncation=True).to(device)
    outputs = model.generate(inputs["input_ids"], max_length=3000)

    sql_query = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return sql_query    

### Basic prompt

We want to convert this example question to an SQL Query.  
Let's use directly BART, without specific informations about the databases.

In [59]:
zeroshot_prompt = """
You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You will only answer with the Query.

<<<
Question: {question}
SQL Query:
>>>
""".strip()

Test the code:

In [60]:
prompt = zeroshot_prompt.format(question=question)
answer = generate(prompt)

print("##### Example 0 #####")
print(f"# {prompt}")
print(f"# {answer}")
print(f"# {gold_sql}")

##### Example 0 #####
# You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You will only answer with the Query.

<<<
Question: How many heads of the departments are older than 56 ?
SQL Query:
>>>
# SELECT T2.professor FROM Database WHERE T1.database_id  =  "SELECT T3.query_id FROM Database"
# SELECT count(*) FROM head WHERE age  >  56


### Detailed prompt

Since this methods give no intresting results, we will try with a more detailed prompt, based on the inforamtions that we retrieved from the databases (Schema...)

In [61]:
db_id = example['db_id']
infos = db_infos.get(db_id)

schemas, p_keys, f_keys = format_schema(infos)

In [62]:
zeroshot_detail_prompt = """
You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You are workign with the database {db_id}, with the following informations:

Schemas:
{schemas}

Primary Keys:
{p_keys}

Foreign Keys:
{f_keys}

You will only answer with the Query.

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [63]:
prompt = zeroshot_detail_prompt.format(
    question=question, 
    db_id=db_id,
    schemas=schemas,
    p_keys=p_keys,
    f_keys=f_keys
)
answer = generate(prompt)

print("##### Example 0 #####")
print(f"# {prompt}")
print(f"# {answer}")
print(f"# {gold_sql}")

##### Example 0 #####
# You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You are workign with the database department_management, with the following informations:

Schemas:
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
head : head_ID (number) , name (text) , born_state (text) , age (number)
management : department_ID (number) , head_ID (number) , temporary_acting (text)

Primary Keys:
department : Department_ID
head : head_ID
management : department_ID

Foreign Keys:
management : head_ID equals head : head_ID
management : department_ID equals department : Department_ID

You will only answer with the Query.

<<<
Question: How many heads of the departments are older than 56 ?
SQL Query:
>>>
# SELECT T1.name ,  T2.name FROM Database AS T1 JOIN department_management AS T2 ON T1(DISTINCT department_id)  =  T3.department_id WHERE T2(T2.Departme

This zeroshot method do not seem adapted with BART.
Let's try it with few-shot.

## FEW SHOT

In [64]:
def format_demo(demo):
    return f"Question: {demo['question']}\nSQL Query: {demo['query']}."
    
def format_demos(demos):
    formatted_str = []
    for demo in demos:
        formatted_str.append(format_demo(demo))
    return '\n\n'.join(formatted_str)
        
def get_random_demo(k, train=train_set):
    examples = []
    for i in range(k):
        examples.append(random.choice(train))
    return examples

### Basic Prompt

In [65]:
zeroshot_prompt = """
You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You will only answer with the Query.

####
Here are some examples:

{examples}
####

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [66]:
demos = format_demos(get_random_demo(5))

prompt = zeroshot_prompt.format(examples=demos, question=question)
answer = generate(prompt)

print("##### Example 0 #####")
print(f"# {prompt}")
print(f"# {answer}")
print(f"# {gold_sql}")

##### Example 0 #####
# You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You will only answer with the Query.

####
Here are some examples:

Question: Who performed the song named "Le Pop"?
SQL Query: SELECT T2.firstname ,  T2.lastname FROM Performance AS T1 JOIN Band AS T2 ON T1.bandmate  =  T2.id JOIN Songs AS T3 ON T3.SongId  =  T1.SongId WHERE T3.Title  =  "Le Pop".

Question: Tell me the types of the policy used by the customer named "Dayana Robel".
SQL Query: SELECT DISTINCT t3.policy_type_code FROM customers AS t1 JOIN customers_policies AS t2 ON t1.customer_id  =  t2.customer_id JOIN available_policies AS t3 ON t2.policy_id  =  t3.policy_id WHERE t1.customer_name  =  "Dayana Robel".

Question: What are the different ids and names of the stations that have had more than 12 bikes available?
SQL Query: SELECT DISTINCT T1.id ,  T1.name FROM station AS T1 JOIN status AS T2 ON T1.id  =  T2.station_id WHERE T2.bikes_available  >  1

### Detailed Prompt

In [67]:
zeroshot_detail_prompt = """
You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You are workign with the database {db_id}, with the following informations:

Schemas:
{schemas}

Primary Keys:
{p_keys}

Foreign Keys:
{f_keys}

You will only answer with the Query.

####
Here are some examples:

{examples}
####

<<<
Question: {question}
SQL Query:
>>>
""".strip()

In [69]:
demos = format_demos(get_random_demo(5))

prompt = zeroshot_detail_prompt.format(
    db_id=db_id,
    schemas=schemas,
    p_keys=p_keys,
    f_keys=f_keys,
    examples=demos,
    question=question, 
)
answer = generate(prompt)

print("##### Example 0 #####")
print(f"# {prompt}")
print(f"# {answer}")
print(f"# {gold_sql}")

##### Example 0 #####
# You are an expert in databases. Your task is to convert question after <<<>>> into a valid SQL Query.

You are workign with the database department_management, with the following informations:

Schemas:
department : Department_ID (number) , Name (text) , Creation (text) , Ranking (number) , Budget_in_Billions (number) , Num_Employees (number)
head : head_ID (number) , name (text) , born_state (text) , age (number)
management : department_ID (number) , head_ID (number) , temporary_acting (text)

Primary Keys:
department : Department_ID
head : head_ID
management : department_ID

Foreign Keys:
management : head_ID equals head : head_ID
management : department_ID equals department : Department_ID

You will only answer with the Query.

####
Here are some examples:

Question: What are the descriptions of the categories that products with product descriptions that contain the letter t are in?
SQL Query: SELECT T1.product_category_description FROM ref_product_categories